## Advanced Firestore Queries

# Introduction to Querying in Firestore

Welcome back! In previous lessons, you learned how to retrieve individual documents and multiple documents from Firestore collections. Today, we'll explore more advanced ways to retrieve data by using queries and filters.

These techniques allow you to efficiently find documents that match specific criteria, making it easier to work with large and complex datasets. For our examples, we'll use a `books` collection, where each document represents a book with fields such as `year`, `title`, and `author`.

---

## Firestore Querying Model Overview

Firestore provides a powerful querying model that lets you filter documents within a collection based on field values. You can retrieve all documents that match one or more conditions, such as finding all books published in a certain year or by a specific author. Queries in Firestore are always performed within a single collection and can use a variety of comparison operators.

Unlike some databases, Firestore does not have a "scan" operation that reads every document in a collection. Instead, it uses pre-computed indexes to retrieve only the documents that match your specified conditions, which ensures performance remains efficient and scalable regardless of your dataset size.

---

## Crafting and Enhancing Queries

To create queries in Firestore, you use method chaining to add filters and select specific fields. Here are some common ways to build and execute queries:

### Basic Query: Filtering by a Single Field

To retrieve all books published in a specific year, use the `where()` method combined with a `FieldFilter`:

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

db = firestore.Client()
books_ref = db.collection('books')

# Define and stream the query
query = books_ref.where(filter=FieldFilter('year', '==', 2018))
results = query.stream()

for doc in results:
    print(doc.id, doc.to_dict())

```

**Example Output:**

> `book_001 {'title': 'The Silent Patient', 'author': 'Alex Michaelides', 'year': 2018, 'genre': 'thriller'}`
> `book_003 {'title': 'Educated', 'author': 'Tara Westover', 'year': 2018, 'genre': 'memoir'}`
> `book_007 {'title': 'Becoming', 'author': 'Michelle Obama', 'year': 2018, 'genre': 'autobiography'}`

### Querying with Multiple Conditions

You can chain multiple `where()` calls to filter documents by more than one field. For example, to find all books published in 2015 with a title that starts with "The":

```python
# Range query on strings to simulate a "begins with" filter
query = books_ref.where(filter=FieldFilter('year', '==', 2015)) \
                 .where(filter=FieldFilter('title', '>=', 'The')) \
                 .where(filter=FieldFilter('title', '<', 'Thf'))

results = query.stream()
for doc in results:
    print(doc.id, doc.to_dict())

```

**Example Output:**

> `book_012 {'title': 'The Girl on the Train', 'author': 'Paula Hawkins', 'year': 2015, 'genre': 'thriller'}`
> `book_018 {'title': 'The Martian', 'author': 'Andy Weir', 'year': 2015, 'genre': 'science fiction'}`

> 📌 **Note:** Firestore does not support "begins with" queries directly, but you can use range queries on strings (incrementing the last character) to approximate this behavior.

### Selecting Specific Fields (Projection)

To minimize data transfer overhead across the network, use the `select()` method to return only the fields your application needs:

```python
query = books_ref.where(filter=FieldFilter('year', '==', 2015)).select(['year', 'title', 'author'])
results = query.stream()

for doc in results:
    print(doc.id, doc.to_dict())

```

**Example Output:**

> `book_012 {'year': 2015, 'title': 'The Girl on the Train', 'author': 'Paula Hawkins'}`
> `book_015 {'year': 2015, 'title': 'Go Set a Watchman', 'author': 'Harper Lee'}`
> `book_018 {'year': 2015, 'title': 'The Martian', 'author': 'Andy Weir'}`

---

## Using Comparison Operators

Firestore supports several comparison operators to evaluate document properties:

| Operator | Description |
| --- | --- |
| `==` | Equal to |
| `!=` | Not equal to |
| `<`, `<=`, `>`, `>=` | Less than, less than or equal to, greater than, greater than or equal to |
| `in` | Field matches any value in a provided list (up to 10 values) |
| `array_contains` | Array field contains a specific scalar value |

### Example: Using the `in` Operator

To find all books published in either 2017 or 2018:

```python
query = books_ref.where(filter=FieldFilter('year', 'in', [2017, 2018]))
results = query.stream()

for doc in results:
    print(doc.id, doc.to_dict())

```

**Example Output:**

> `book_001 {'title': 'The Silent Patient', 'author': 'Alex Michaelides', 'year': 2018, 'genre': 'thriller'}`
> `book_003 {'title': 'Educated', 'author': 'Tara Westover', 'year': 2018, 'genre': 'memoir'}`
> `book_005 {'title': "The Handmaid's Tale", 'author': 'Margaret Atwood', 'year': 2017, 'genre': 'dystopian'}`

---

## Limitations and Considerations

* **Query Result Size:** A single query stream chunk can safely handle data scaling, but large datasets should utilize pagination to optimize client-side memory performance.
* **Index Requirements:** Simple single-field queries are indexed automatically. However, compound queries (filtering across multiple fields with inequalities or sorting) require a composite index. If an index is missing, the Firestore SDK raises an exception containing a direct URL link to generate it in the Google Cloud Console.
* **Single Collection Scope:** Standard queries run within the boundary of a single collection. You cannot perform SQL-like relational `JOIN` operations across different collections natively.
* **Inequality Filters:** If you include inequality filters (`!=`, `<`, `>`, `>=`), any other field filters containing inequalities must target that exact same field.
* **No Collection Scans:** Firestore enforces strict query planning; it does not support brute-force database scans without defined constraints, ensuring execution speeds stay lightning-fast.

---

## Summary and Looking Ahead

In this lesson, you mastered Firestore's core querying engine:

* Filtering documents with single and chained compound `FieldFilter` conditions.
* Optimizing network bandwidth with field projection using `.select()`.
* Evaluating datasets with operators like `==`, inequalities, and membership checks (`in`).

Practice designing various query combinations to master data indexing behaviors. This foundation prepares you for advanced schema manipulation and high-performance querying architectures!

## Executing Firestore Query and Scan Operations

This task is designed to familiarize you with executing Firestore Query operations using a provided Python script. You will run the script that interacts with a pre-configured Movies collection, which includes operations to retrieve data using various Query configurations. Observe how data is retrieved under different scenarios, including using simple conditions, field selection, and multiple filter expressions. This exercise will help enhance your understanding of data retrieval dynamics in Firestore.

Important Note: Running scripts can modify resources in our GCP simulator. To revert to the initial state, use the reset button located in the top right corner. Resetting will erase any code changes, so copy your code to the clipboard to preserve it during a reset.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# Initialize Firestore client
db = firestore.Client()

# Create reference to Movies collection
movies_collection = db.collection('Movies')

# Populate collection with data
movies_collection.add({'year': 2016, 'title': 'The Big New Movie'})
movies_collection.add({'year': 2017, 'title': 'The Bigger, Newer Movie'})
movies_collection.add({'year': 2017, 'title': 'Yet Another Movie'})
movies_collection.add({'year': 2017, 'title': 'One More Movie'})
movies_collection.add({'year': 2015, 'title': 'An Old Movie'})
movies_collection.add({'year': 2018, 'title': 'Another New Movie'})

# Example with Query command 1: Simple Query
query_1 = movies_collection.where(filter=FieldFilter('year', '==', 2018))
query_1_results = [doc.to_dict() for doc in query_1.stream()]
print("Query 1 Results:", query_1_results)

# Example with Query command 2: Query with field selection
query_2 = movies_collection.where(filter=FieldFilter('year', '==', 2015)).select(['year', 'title'])
query_2_results = [doc.to_dict() for doc in query_2.stream()]
print("Query 2 Results:", query_2_results)

# Example with Query command 3: Retrieve all documents from collection
query_3 = movies_collection
query_3_results = [doc.to_dict() for doc in query_3.stream()]
print("Query 3 Results (All Documents):", query_3_results)

# Example with Query command 4: Multiple condition query
query_4 = movies_collection.where(filter=FieldFilter('year', 'in', [2017, 2018]))
query_4_results = [doc.to_dict() for doc in query_4.stream()]
print("Query 4 Results (Multiple Years):", query_4_results)

```

Running this script highlights the fundamental ways to extract data from a NoSQL database. Rather than searching through every single row like a traditional relational scan, Firestore relies entirely on backend indexes, allowing these queries to return data instantly regardless of the size of your collection.

Here is an architectural breakdown of what happens behind the scenes for each query configuration.

---

## 1. Document Creation Twist: `add()` vs `set()`

Unlike previous scripts where you used `.document('1').set(...)`, this script uses `movies_collection.add(...)`.

* When you call `.add()`, you don't provide a document ID.
* Firestore automatically assigns a randomized, unique 20-character alphanumeric hash string (e.g., `qW3b7RxlPZ91KjM...`) to act as the key field.

---

## 2. Breaking Down the Query Configurations

### Query 1: The Single Equality Filter

```python
query_1 = movies_collection.where(filter=FieldFilter('year', '==', 2018))

```

* **Mechanics:** Firestore immediately looks up the pre-built internal index for the `year` field. It jumps directly to the entry for `2018` and fetches the corresponding document snapshots.
* **Expected Output:** Returns a list containing one dictionary: `Another New Movie`.

### Query 2: Field Selection (Projection Optimization)

```python
query_2 = movies_collection.where(filter=FieldFilter('year', '==', 2015)).select(['year', 'title'])

```

* **Mechanics:** While this looks similar to Query 1, the trailing `.select([...])` alters the server pipeline. The Firestore server extracts the document, strips out any non-specified fields (such as `genre` if it existed), and delivers a lighter payload over the network.
* **Expected Output:** Returns the data for `An Old Movie`, explicitly showing only the specified attributes.

### Query 3: Streaming the Entire Collection

```python
query_3 = movies_collection
query_3_results = [doc.to_dict() for doc in query_3.stream()]

```

* **Mechanics:** Calling `.stream()` directly on a collection reference retrieves all records currently present in that segment. Firestore pulls these documents via a streaming RPC call, allowing your client app to handle entries continuously as they arrive over the wire.
* **Expected Output:** A collection of all 6 generated movie documents.

### Query 4: Set Membership with the `in` Operator

```python
query_4 = movies_collection.where(filter=FieldFilter('year', 'in', [2017, 2018]))

```

* **Mechanics:** The `in` operator lets you pass a list of up to 10 values. Under the hood, Firestore splits this into parallel sub-queries (`year == 2017` and `year == 2018`) and merges the resulting streams seamlessly before handing them back to your code.
* **Expected Output:** Returns 4 records: the three movies from 2017 alongside the single movie from 2018.

---

## Summary of Query Behaviors

| Feature | Method | Network Payload | Best For... |
| --- | --- | --- | --- |
| **Direct Filter** | `.where(filter=...)` | Full Document Map | Basic target searches. |
| **Field Projection** | `.select([...])` | Targeted Key/Value Only | Mobile or web views needing small data chunks. |
| **Collection Stream** | `.stream()` on Collection | Entire Collection | Backups, reporting, or localized cache syncs. |
| **Set Containment** | `'in'` operator | Combined Query Matches | Matching against a specific group of IDs or tags. |

## Firestore Compound Queries with Multiple Conditions

## Firestore Compound Where Queries with Multiple Filter Conditions

## Firestore Where Clause Query Implementation

## Implementing Firestore Querying Operations for Movies